##IMPORTAR BIBLIOTECAS

In [ ]:
!pip install -q beautifulsoup4 requests numpy pandas plotly scipy lxml

import os
import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
import math
import time
from scipy.signal import savgol_filter
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import drive


## CONFIGURAR FILTROS

In [ ]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

NOME_ARQUIVO = "Rotas de São Paulo, SP até Santiago, Chile.kml"

# --- DEFINIR TRECHO ---
KM_INICIO = 2800
KM_FIM = 3200
FATIA_VISUAL = 100 # Linhas verticais no gráfico a cada 50km

# Busca Arquivo
caminhos = [
    f"/content/drive/MyDrive/{NOME_ARQUIVO}",
    f"/content/drive/MyDrive/analisededados/setr/Projeto_2025/{NOME_ARQUIVO}",
    f"/content/drive/My Drive/{NOME_ARQUIVO}"
]
ARQUIVO_KML = None
for c in caminhos:
    if os.path.exists(c): ARQUIVO_KML = c; break

if not ARQUIVO_KML:
    # Persistir na Busca
    for root, dirs, files in os.walk("/content/drive/MyDrive"):
        for f in files:
            if f == NOME_ARQUIVO: ARQUIVO_KML = os.path.join(root, f); break
        if ARQUIVO_KML: break

if not ARQUIVO_KML: raise FileNotFoundError("Arquivo não encontrado.")
print(f" Arquivo: {ARQUIVO_KML}")

Mounted at /content/drive
 Arquivo: /content/drive/MyDrive/Rotas de São Paulo, SP até Santiago, Chile.kml


## LEITURA GEOMÉTRICA

In [ ]:
print("1 Lendo a rota inteira para calibrar o odômetro...")

try:
    with open(ARQUIVO_KML, 'r', encoding='utf-8') as f: soup = BeautifulSoup(f, 'xml')
except:
    with open(ARQUIVO_KML, 'r') as f: soup = BeautifulSoup(f, 'xml')

coords_text = soup.text.split()
lats_raw, lons_raw = [], []

for item in coords_text:
    if ',' in item:
        try:
            parts = item.split(',')
            lon = float(parts[0]); lat = float(parts[1])
            if -90 <= lat <= 90: lons_raw.append(lon); lats_raw.append(lat)
        except: pass

# Amostragem leve
step = 2 if len(lats_raw) > 5000 else 1
lats_full = np.array(lats_raw[::step])
lons_full = np.array(lons_raw[::step])

# CÁLCULO DE DISTÂNCIA TOTAL
def haversine_vec(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlam/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

dists = [0]
for i in range(1, len(lats_full)):
    d = haversine_vec(lats_full[i-1], lons_full[i-1], lats_full[i], lons_full[i])
    if d > 20000: d = 0
    dists.append(dists[-1] + d)

dist_arr_full = np.array(dists) / 1000 # Km

print(f"   -> Rota Total Calculada: {dist_arr_full[-1]:.0f} km")

1 Lendo a rota inteira para calibrar o odômetro...
   -> Rota Total Calculada: 3193 km


## RECORTE DO TRECHO

In [ ]:
print(f"2 Recortando rota do Km {KM_INICIO} ao Km {KM_FIM}...")

mask = (dist_arr_full >= KM_INICIO) & (dist_arr_full <= KM_FIM)

if not np.any(mask):
    raise ValueError(f"❌ O trecho {KM_INICIO}-{KM_FIM} está fora da rota.")

lats_trecho = lats_full[mask]
lons_trecho = lons_full[mask]
dist_trecho = dist_arr_full[mask]

print(f"   -> Pontos para Análise: {len(lats_trecho)}")


2 Recortando rota do Km 2800 ao Km 3200...
   -> Pontos para Análise: 1822


## ENGENHARIA GEOESPACIAL

In [ ]:
def get_elevation_safe(lats, lons):
    url = "https://api.open-meteo.com/v1/elevation"
    elevacoes = []
    chunk = 50
    print(f"3 Baixando Altitudes REAIS (Com protocolo Anti-Bloqueio)...")

    total_chunks = len(lats) // chunk + 1

    for i in range(0, len(lats), chunk):
        c_lats = lats[i:i+chunk]; c_lons = lons[i:i+chunk]
        params = {"latitude": ",".join(map(str, c_lats)), "longitude": ",".join(map(str, c_lons))}

        # --- LÓGICA DE PERSISTÊNCIA ---
        sucesso = False
        wait_time = 60 # Começa com 60 segundos (O PEDIDO)

        while not sucesso:
            try:
                r = requests.get(url, params=params, timeout=15)

                if r.status_code == 200:
                    d = r.json()['elevation']
                    last = elevacoes[-1] if elevacoes else 0
                    clean = [x if x is not None else last for x in d]
                    elevacoes.extend(clean)
                    sucesso = True
                    # Se deu certo, reseta o tempo para o próximo lote (opcional, mas bom)
                    wait_time = 60

                elif r.status_code == 429:
                    # AQUI ESTÁ A LÓGICA DE ESCALADA
                    print(f"   ⛔ BLOQUEIO 429 DETECTADO! Esperando {wait_time} segundos...")
                    time.sleep(wait_time)
                    wait_time = wait_time * 3 # TRIPLICA o tempo para a próxima tentativa
                    print(f"   🔄 Tentando novamente agora...")

                else:
                    # Outros erros (500, etc)
                    print(f"   ⚠️ Erro {r.status_code}. Esperando 5s...")
                    time.sleep(5)
            except:
                print("   ⚠️ Erro de conexão. Esperando 5s...")
                time.sleep(5)

        # Feedback visual de progresso
        if i % 200 == 0:
            print(f"   -> Lote processado: {i}/{len(lats)}")

        time.sleep(0.5) # Pausa mínima padrão

    return np.array(elevacoes)

# Baixa Altitude
alts_trecho = get_elevation_safe(lats_trecho, lons_trecho)
alts_plot = savgol_filter(alts_trecho, 15, 3) if len(alts_trecho) > 15 else alts_trecho

# Calcula Sinuosidade
sinuosidade = []
window = 10
for i in range(len(lats_trecho)):
    s = max(0, i - window); e = min(len(lats_trecho) - 1, i + window)
    if e - s < 2: sinuosidade.append(1.0); continue

    real = (dist_trecho[e] - dist_trecho[s]) * 1000
    reta = haversine_vec(lats_trecho[s], lons_trecho[s], lats_trecho[e], lons_trecho[e])
    sinuosidade.append(real/reta if reta > 0 else 1.0)
sin_smooth = savgol_filter(sinuosidade, 21, 2)

3 Baixando Altitudes REAIS (Com protocolo Anti-Bloqueio)...
   -> Lote processado: 0/1822
   -> Lote processado: 200/1822
   -> Lote processado: 400/1822
   -> Lote processado: 600/1822
   -> Lote processado: 800/1822
   ⛔ BLOQUEIO 429 DETECTADO! Esperando 60 segundos...
   🔄 Tentando novamente agora...
   -> Lote processado: 1000/1822
   -> Lote processado: 1200/1822
   -> Lote processado: 1400/1822
   ⛔ BLOQUEIO 429 DETECTADO! Esperando 60 segundos...
   🔄 Tentando novamente agora...
   -> Lote processado: 1600/1822
   -> Lote processado: 1800/1822


## ALTIMETRIA E SINUOSIDADE

In [ ]:
print("4 Gerando Gráfico...")

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Altitude
fig.add_trace(go.Scatter(
    x=dist_trecho, y=alts_plot, mode='lines', fill='tozeroy',
    name='Altitude (m)', line=dict(color='rgb(34, 139, 34)', width=1.5),
    fillcolor='rgba(34, 139, 34, 0.3)',
    hovertemplate='Km %{x:.1f}<br>Alt: %{y:.0f}m<extra></extra>'
), secondary_y=False)

# Sinuosidade
fig.add_trace(go.Scatter(
    x=dist_trecho, y=sin_smooth, mode='lines', name='Sinuosidade',
    line=dict(color='rgb(220, 20, 60)', width=2),
    hovertemplate='Sinuosidade: %{y:.2f}<extra></extra>'
), secondary_y=True)

# Marcadores Visuais
inicio_grade = int(dist_trecho[0] // FATIA_VISUAL) * FATIA_VISUAL
if inicio_grade < dist_trecho[0]: inicio_grade += FATIA_VISUAL

for km_corte in range(inicio_grade, int(dist_trecho[-1]) + 1, FATIA_VISUAL):
    fig.add_vline(x=km_corte, line_width=1, line_dash="dash", line_color="black")
    fig.add_annotation(
        x=km_corte, y=max(alts_plot),
        text=f"{km_corte}km", showarrow=False, yshift=10, bgcolor="yellow", bordercolor="black"
    )

fig.update_layout(
    title=f"<b>Raio-X de Trecho: Km {KM_INICIO} ao {KM_FIM}</b>",
    hovermode="x unified", plot_bgcolor='white', height=600,
    legend=dict(orientation="h", y=1.02, x=0.5, xanchor="center")
)
fig.update_yaxes(title="Altitude (m)", secondary_y=False, showgrid=True, gridcolor='#eee')
fig.update_yaxes(title="Sinuosidade", secondary_y=True, showgrid=False, range=[1.0, 3.5])
fig.update_xaxes(title="Distância (km)", showgrid=True)

fig.show()

4 Gerando Gráfico...


## CÁLCULO E ANÁLISE DE INCLINAÇÃO (GRADE %)

In [ ]:

print("5️⃣ Calculando Inclinação: Do Ponto Atual para a Frente...")

if 'alts_trecho' not in globals() or 'dist_trecho' not in globals():
    print("❌ ERRO: Variáveis não encontradas.")
else:
    import numpy as np
    import plotly.graph_objects as go
    from scipy.signal import savgol_filter

    # --- 1. CONFIGURAÇÃO DA VISÃO DO MOTORISTA ---
    VISAO_METROS = 1000 # Analisando os próximos 1000 metros

    # Suavização LEVE na altitude apenas para tirar ruído de GPS
    alts_clean = savgol_filter(alts_trecho, 11, 3)

    grade_forward = []
    dist_m = dist_trecho * 1000
    total_pontos = len(dist_m)

    # --- 2. LOOP "FAROL ALTO" (Olha só pra frente) ---
    for i in range(total_pontos):
        idx_atual = i
        idx_futuro = i
        while idx_futuro < total_pontos - 1 and (dist_m[idx_futuro] - dist_m[idx_atual]) < VISAO_METROS:
            idx_futuro += 1

        if idx_futuro == idx_atual:
            grade = 0 if len(grade_forward) == 0 else grade_forward[-1]
        else:
            d_alt = alts_clean[idx_futuro] - alts_clean[idx_atual]
            d_dist = dist_m[idx_futuro] - dist_m[idx_atual]
            grade = (d_alt / d_dist) * 100
        grade_forward.append(grade)

    grade_forward = np.array(grade_forward)
    grade_smooth = grade_forward

    # --- 3. VALIDAÇÃO NO KM 3133.7 (COM ALTITUDES A e B) ---
    # 1. Acha o índice do ponto A (atual)
    idx_a = (np.abs(dist_trecho - 3133.7)).argmin()

    # 2. Acha o índice do ponto B (500m à frente) usando a mesma lógica do loop
    idx_b = idx_a
    while idx_b < total_pontos - 1 and (dist_m[idx_b] - dist_m[idx_a]) < VISAO_METROS:
        idx_b += 1

    # 3. Resgata os valores para o print
    alt_a = alts_clean[idx_a]
    alt_b = alts_clean[idx_b]
    km_a = dist_trecho[idx_a]
    km_b = dist_trecho[idx_b]
    val_incl = grade_forward[idx_a]

    print(f"\n📊 RESULTADO DA VISÃO ({VISAO_METROS}m à frente):")
    print(f"   📍 Ponto A (Início): Km {km_a:.3f} | Altitude: {alt_a:.1f} m")
    print(f"   🏁 Ponto B (Fim):    Km {km_b:.3f} | Altitude: {alt_b:.1f} m")
    print(f"   📉 Desnível: {alt_b - alt_a:.1f} metros")
    print(f"   📐 Inclinação Calculada: {val_incl:.2f}%")

    # --- 4. GRÁFICO ---
    fig_grade = go.Figure()
    fig_grade.add_trace(go.Scatter(
        x=dist_trecho, y=grade_forward,
        mode='lines', name='Inclinação Futura (%)',
        line=dict(color='black', width=1.5),
        hovertemplate='Estou no Km %{x:.1f}<br>Próximos 1000m: <b>%{y:.2f}%</b><extra></extra>'
    ))

    fig_grade.add_hline(y=6, line_dash="dot", line_color="red", annotation_text="Sobe Muito (+6%)")
    fig_grade.add_hline(y=-6, line_dash="dot", line_color="blue", annotation_text="Desce Muito (-6%)")

    fig_grade.update_layout(
        title=f"<b>Inclinação Prospectiva (O que vem pela frente nos próximos {VISAO_METROS}m)</b>",
        yaxis_title="Inclinação (%)", xaxis_title="Distância (km)",
        plot_bgcolor='white', height=500, hovermode="x unified"
    )
    fig_grade.update_yaxes(showgrid=True, gridcolor='#eee', range=[-18, 18])
    fig_grade.show()

5️⃣ Calculando Inclinação: Do Ponto Atual para a Frente...

📊 RESULTADO DA VISÃO (1000m à frente):
   📍 Ponto A (Início): Km 3133.681 | Altitude: 817.1 m
   🏁 Ponto B (Fim):    Km 3134.770 | Altitude: 776.0 m
   📉 Desnível: -41.1 metros
   📐 Inclinação Calculada: -3.77%


/tmp/ipython-input-2676355802.py:32: RuntimeWarning:

divide by zero encountered in scalar divide



## CÁLCULO E ANÁLISE DE ENERGIA E DEMANDA DE FREIO

In [ ]:
print("6️⃣ Calculando Física do Caminhão (Potência Gravitacional)...")

if 'grade_smooth' not in globals():
    print("❌ ERRO: Variável 'grade_smooth' não encontrada. Rode a célula de INCLINAÇÃO primeiro.")
else:
    # --- 1. CONFIGURAÇÃO DO VEÍCULO ---
    MASSA_TOTAL_TON = 50    # Peso Bruto Total (Caminhão + Carga)
    VELOCIDADE_REF_KMH = 20  # Velocidade de referência para o cálculo (Segurança)
    LIMITE_FREIO_MOTOR_KW = 470 # Potência média de um freio motor (VEB/Retarder)

    # --- 2. CÁLCULOS FÍSICOS ---
    # Conversão de unidades
    massa_kg = MASSA_TOTAL_TON * 1000
    veloc_ms = VELOCIDADE_REF_KMH / 3.6
    gravidade = 9.81 # m/s²

    # Fórmula: Potência (Watts) = Força * Velocidade
    # Força Gravitacional na Rampa = Massa * g * sen(teta)


    # Vetor de Potência (kW)
    # Positivo = Motor fazendo força (Subida)
    # Negativo = Freio dissipando energia (Descida)
    potencia_kw = (massa_kg * gravidade * (grade_smooth / 100) * veloc_ms) / 1000

    # Separação para o Gráfico
    demanda_motor = np.where(potencia_kw > 0, potencia_kw, 0)
    demanda_freio = np.where(potencia_kw < 0, np.abs(potencia_kw), 0) # Módulo para plotar positivo

    # Estatísticas
    pico_freio = np.max(demanda_freio)
    pico_motor = np.max(demanda_motor)

    print(f"\n ANÁLISE PARA {MASSA_TOTAL_TON} TONELADAS A {VELOCIDADE_REF_KMH} KM/H:")
    print(f"    Calor Máximo no Freio: {pico_freio:.0f} kW (Pico de Exigência)")

    if pico_freio > LIMITE_FREIO_MOTOR_KW:
        print(f"   ⚠️ ALERTA CRÍTICO: Existem pontos exigindo {pico_freio:.0f} kW.")
        print(f"      Isso supera o freio motor ({LIMITE_FREIO_MOTOR_KW} kW). Risco de Fading!")
    else:
        print(f"   ✅ O freio motor suporta a descida (Máx {pico_freio:.0f} kW).")

    # --- 3. GRÁFICO DE ENERGIA ---
    fig_fisica = go.Figure()

    # Área Verde (Motor/Subida)
    fig_fisica.add_trace(go.Scatter(
        x=dist_trecho, y=demanda_motor,
        mode='lines', fill='tozeroy', name='Exigência Motor (Tração)',
        line=dict(color='green', width=1),
        hovertemplate='Subida<br>Motor: %{y:.0f} kW<extra></extra>'
    ))

    # Área Vermelha (Freio/Descida)
    fig_fisica.add_trace(go.Scatter(
        x=dist_trecho, y=demanda_freio,
        mode='lines', fill='tozeroy', name='Exigência Freio (Dissipação)',
        line=dict(color='red', width=1),
        hovertemplate='Descida<br>Freio: %{y:.0f} kW<extra></extra>'
    ))

    # Linha de Limite do Freio Motor (Referência)
    fig_fisica.add_hline(
        y=LIMITE_FREIO_MOTOR_KW,
        line_dash="dash", line_color="black",
        annotation_text=f"Limite Freio Motor ({LIMITE_FREIO_MOTOR_KW} kW)",
        annotation_position="top left"
    )

    fig_fisica.update_layout(
        title=f"<b>Análise de Energia e Eficiência Freio Motor: {MASSA_TOTAL_TON} Ton</b>",
        yaxis_title="Potência Exigida (kW)",
        xaxis_title="Distância (km)",
        plot_bgcolor='white',
        height=500,
        hovermode="x unified"
    )
    fig_fisica.update_xaxes(showgrid=True, gridcolor='#eee')

    # --- AJUSTE SOLICITADO: ESCALA TRAVADA EM 2000 ---
    fig_fisica.update_yaxes(
        showgrid=True,
        gridcolor='#eee',
        range=[0, 2000] # <--- Trava de escala aplicada aqui
    )

    fig_fisica.show()

6️⃣ Calculando Física do Caminhão (Potência Gravitacional)...

 ANÁLISE PARA 74 TONELADAS A 20 KM/H:
    Calor Máximo no Freio: 776 kW (Pico de Exigência)
   ⚠️ ALERTA CRÍTICO: Existem pontos exigindo 776 kW.
      Isso supera o freio motor (470 kW). Risco de Fading!


In [ ]:
print("6️⃣ Calculando Física do Caminhão (Potência de Tração e Subida)...")

if 'grade_smooth' not in globals():
    print("❌ ERRO: Variável 'grade_smooth' não encontrada. Rode a célula de INCLINAÇÃO primeiro.")
else:
    # --- 1. CONFIGURAÇÃO DO VEÍCULO (EDITE AQUI) ---
    MASSA_TOTAL_TON = 74
    VELOCIDADE_REF_KMH = 20
    LIMITE_FREIO_MOTOR_KW = 470
    POTENCIA_MOTOR_HP = 540     # Cavalaria do motor (Ex: Volvo FH 540)

    # Conversão de HP para kW (1 HP ≈ 0.745 kW)
    LIMITE_MOTOR_TRAÇÃO_KW = POTENCIA_MOTOR_HP * 0.745

    # --- 2. CÁLCULOS FÍSICOS ---
    massa_kg = MASSA_TOTAL_TON * 1000
    veloc_ms = VELOCIDADE_REF_KMH / 3.6
    gravidade = 9.81

    # Vetor de Potência (kW)
    potencia_kw = (massa_kg * gravidade * (grade_smooth / 100) * veloc_ms) / 1000

    demanda_motor = np.where(potencia_kw > 0, potencia_kw, 0)
    demanda_freio = np.where(potencia_kw < 0, np.abs(potencia_kw), 0)

    pico_motor = np.max(demanda_motor)
    pico_freio = np.max(demanda_freio)

    print(f"\n🚀 ANÁLISE DE SUBIDA: {MASSA_TOTAL_TON} TON A {VELOCIDADE_REF_KMH} KM/H")
    print(f"   ⚡ Potência Máxima Exigida na Subida: {pico_motor:.0f} kW")
    print(f"   💪 Capacidade do Motor: {LIMITE_MOTOR_TRAÇÃO_KW:.0f} kW ({POTENCIA_MOTOR_HP} HP)")

    if pico_motor > LIMITE_MOTOR_TRAÇÃO_KW:
        print(f"   ⚠️ ALERTA DE SUBIDA: O motor NÃO mantém {VELOCIDADE_REF_KMH} km/h nas rampas mais fortes!")
        print(f"      A velocidade vai cair (perda de performance).")
    else:
        print(f"   ✅ O motor vence as subidas mantendo os {VELOCIDADE_REF_KMH} km/h.")

    # --- 3. GRÁFICO DE ENERGIA ---
    fig_fisica = go.Figure()

    # Área Verde (Motor/Subida)
    fig_fisica.add_trace(go.Scatter(
        x=dist_trecho, y=demanda_motor,
        mode='lines', fill='tozeroy', name='Demanda de Tração (Motor)',
        line=dict(color='green', width=1.5),
        hovertemplate='Subida<br>Motor: %{y:.0f} kW<extra></extra>'
    ))

    # Área Vermelha (Freio/Descida)
    fig_fisica.add_trace(go.Scatter(
        x=dist_trecho, y=demanda_freio,
        mode='lines', fill='tozeroy', name='Demanda de Frenagem (VEB)',
        line=dict(color='red', width=1.5),
        hovertemplate='Descida<br>Freio: %{y:.0f} kW<extra></extra>'
    ))

    # LINHA DE LIMITE: POTÊNCIA DO MOTOR (TRAÇÃO)
    fig_fisica.add_hline(
        y=LIMITE_MOTOR_TRAÇÃO_KW,
        line_dash="dot", line_color="green",
        annotation_text=f"Limite Motor ({POTENCIA_MOTOR_HP} HP)",
        annotation_position="top right"
    )

    # LINHA DE LIMITE: FREIO MOTOR
    fig_fisica.add_hline(
        y=LIMITE_FREIO_MOTOR_KW,
        line_dash="dash", line_color="black",
        annotation_text=f"Limite VEB ({LIMITE_FREIO_MOTOR_KW} kW)",
        annotation_position="top left"
    )

    fig_fisica.update_layout(
        title=f"<b>Análise de Tração e Frenagem: {MASSA_TOTAL_TON} Ton @ {VELOCIDADE_REF_KMH} km/h</b>",
        yaxis_title="Potência (kW)",
        xaxis_title="Distância (km)",
        plot_bgcolor='white',
        height=550,
        hovermode="x unified"
    )

    fig_fisica.update_xaxes(showgrid=True, gridcolor='#eee')

    # ESCALA TRAVADA EM 2000 PARA VISUALIZAÇÃO TÉCNICA
    fig_fisica.update_yaxes(
        showgrid=True,
        gridcolor='#eee',
        range=[0, 2000]
    )

    fig_fisica.show()

6️⃣ Calculando Física do Caminhão (Potência de Tração e Subida)...

🚀 ANÁLISE DE SUBIDA: 74 TON A 20 KM/H
   ⚡ Potência Máxima Exigida na Subida: inf kW
   💪 Capacidade do Motor: 402 kW (540 HP)
   ⚠️ ALERTA DE SUBIDA: O motor NÃO mantém 20 km/h nas rampas mais fortes!
      A velocidade vai cair (perda de performance).


In [ ]:
print("🚀 Gerando Análise de Eficiência e Segurança (Versão Terça-Feira)...")

if 'grade_smooth' not in globals():
    print("❌ ERRO: Variável 'grade_smooth' não encontrada.")
else:
    # --- 1. CONFIGURAÇÃO (AJUSTE AQUI) ---
    MASSA_TOTAL_TON = 20
    VELOCIDADE_REF_KMH = 40
    LIMITE_FREIO_MOTOR_KW = 470
    POTENCIA_MOTOR_HP = 540

    # Conversões Técnicas
    LIMITE_TRAÇÃO_KW = POTENCIA_MOTOR_HP * 0.745
    massa_kg = MASSA_TOTAL_TON * 1000
    veloc_ms = VELOCIDADE_REF_KMH / 3.6
    gravidade = 9.81

    # --- 2. CÁLCULO DA POTÊNCIA ---
    potencia_kw = (massa_kg * gravidade * (grade_smooth / 100) * veloc_ms) / 1000

    # Criando listas para o tooltip personalizado
    texto_tooltip = []
    for p in potencia_kw:
        hp_equiv = abs(p) / 0.745
        if p > 0:
            # Lógica de Consumo na Subida
            status = "ECONÔMICO (Torque)" if p < (LIMITE_TRAÇÃO_KW * 0.6) else "ALTO CONSUMO (Potência)"
            texto_tooltip.append(f"Subida<br>Esforço: {p:.0f} kW ({hp_equiv:.0f} HP)<br>Status: {status}")
        else:
            # Lógica de Segurança na Descida
            seguranca = "SEGURO (VEB)" if abs(p) < LIMITE_FREIO_MOTOR_KW else "PERIGO (Fading)"
            texto_tooltip.append(f"Descida<br>Dissipação: {abs(p):.0f} kW ({hp_equiv:.0f} HP)<br>Status: {seguranca}")

    demanda_motor = np.where(potencia_kw > 0, potencia_kw, 0)
    demanda_freio = np.where(potencia_kw < 0, np.abs(potencia_kw), 0)

    # --- 3. GRÁFICO PROFISSIONAL ---
    fig_master = go.Figure()

    # Subida (Verde)
    fig_master.add_trace(go.Scatter(
        x=dist_trecho, y=demanda_motor,
        mode='lines', fill='tozeroy', name='Tração (Diesel)',
        line=dict(color='green', width=1.5),
        customdata=texto_tooltip,
        hovertemplate='<b>Km %{x:.1f}</b><br>%{customdata}<extra></extra>'
    ))

    # Descida (Vermelha)
    fig_master.add_trace(go.Scatter(
        x=dist_trecho, y=demanda_freio,
        mode='lines', fill='tozeroy', name='Frenagem (Segurança)',
        line=dict(color='red', width=1.5),
        customdata=texto_tooltip,
        hovertemplate='<b>Km %{x:.1f}</b><br>%{customdata}<extra></extra>'
    ))

    # Linhas de Referência
    fig_master.add_hline(y=LIMITE_TRAÇÃO_KW, line_dash="dot", line_color="green",
                         annotation_text="Limite Motor (HP)")
    fig_master.add_hline(y=LIMITE_FREIO_MOTOR_KW, line_dash="dash", line_color="black",
                         annotation_text="Limite Freio Motor (kW)")

    fig_master.update_layout(
        title=f"<b>Raio-X de Energia e Consumo: {MASSA_TOTAL_TON} Ton</b>",
        yaxis_title="Potência (kW)", xaxis_title="Distância (km)",
        plot_bgcolor='white', height=550, hovermode="x unified"
    )

    fig_master.update_yaxes(range=[0, 2000], showgrid=True, gridcolor='#eee')
    fig_master.show()

🚀 Gerando Análise de Eficiência e Segurança (Versão Terça-Feira)...


##  CÁLCULO E ANÁLISE DE ESTABILIDADE: CARGA LÍQUIDA (EFEITO SLOSH)

In [ ]:
print("7️⃣ Calculando Risco de Tombamento (Slosh)...")

import numpy as np
import plotly.graph_objects as go
from scipy.signal import savgol_filter

# --- PARÂMETROS DE SEGURANÇA ---
VELOCIDADE_CURVA_KMH = 40  # Velocidade de análise
LIMITE_G_SOLIDO = 0.40      # Limite Carga Seca
LIMITE_G_LIQUIDO = 0.28     # Limite Carga Líquida

# --- 1. FUNÇÕES DE GEOMETRIA ---
def get_bearing(lat1, lon1, lat2, lon2):
    dLon = np.radians(lon2 - lon1)
    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)
    y = np.sin(dLon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dLon)
    brng = np.degrees(np.arctan2(y, x))
    return (brng + 360) % 360

if 'lats_trecho' not in globals():
    print("❌ ERRO: Rode as células anteriores primeiro.")
else:
    # --- 2. CÁLCULO FÍSICO ---
    # Suavização para evitar ruído
    window_geo = 15
    if len(lats_trecho) > window_geo:
        lats_s = savgol_filter(lats_trecho, window_geo, 3)
        lons_s = savgol_filter(lons_trecho, window_geo, 3)
    else:
        lats_s, lons_s = lats_trecho, lons_trecho

    # Cálculo dos Ângulos
    bearings = []
    for i in range(len(lats_s)-1):
        b = get_bearing(lats_s[i], lons_s[i], lats_s[i+1], lons_s[i+1])
        bearings.append(b)
    bearings.append(bearings[-1])

    # Delta Angle e Distância
    delta_angle = np.diff(bearings, prepend=bearings[0])
    delta_angle = np.where(delta_angle > 180, delta_angle - 360, delta_angle)
    delta_angle = np.where(delta_angle < -180, delta_angle + 360, delta_angle)

    delta_dist_m = np.diff(dist_trecho, prepend=dist_trecho[0]) * 1000
    delta_dist_m[delta_dist_m < 1] = 1

    # Curvatura e Raio
    curvature = np.abs(delta_angle) / delta_dist_m
    raio_curva = np.zeros_like(curvature)
    mask_c = curvature > 0.001
    raio_curva[mask_c] = 57.2958 / curvature[mask_c]
    raio_curva[~mask_c] = 2000
    raio_smooth = savgol_filter(raio_curva, 21, 2)

    # Força G
    v_ms = VELOCIDADE_CURVA_KMH / 3.6
    g_force = (v_ms ** 2) / (raio_smooth * 9.81)
    g_force = np.clip(g_force, 0, 1.5)

    # --- 3. ESTATÍSTICAS ---
    max_g = np.max(g_force)
    print(f"\n🌊 ANÁLISE DE SLOSH ({VELOCIDADE_CURVA_KMH} km/h):")
    print(f"   🌀 Força G Máxima: {max_g:.2f}g")

    if max_g > LIMITE_G_LIQUIDO:
        print(f"   ⚠️ PERIGO: {max_g:.2f}g (Limite Líquido: {LIMITE_G_LIQUIDO}g)")
    else:
        print(f"   ✅ Estável (< {LIMITE_G_LIQUIDO}g)")

    # --- 4. GRÁFICO ---
    fig_slosh = go.Figure()

    # Linha Principal (Laranja)
    fig_slosh.add_trace(go.Scatter(
        x=dist_trecho,
        y=g_force,
        mode='lines',
        name='Aceleração Lateral (g)',
        line=dict(color='#FFA500', width=2),
        fill='tozeroy',
        fillcolor='rgba(255, 165, 0, 0.1)',
        text=raio_smooth,
        # AQUI ESTAVA O ERRO: Agora está em uma linha única e segura
        hovertemplate='Km %{x:.1f}<br>Força: <b>%{y:.2f}g</b><br>Raio: %{text:.0f}m<extra></extra>'
    ))

    # Linha Limite Líquido (Vermelha)
    fig_slosh.add_hline(
        y=LIMITE_G_LIQUIDO,
        line_dash="dash", line_color="firebrick", line_width=2,
        annotation_text=f"Limite Carga LÍQUIDA ({LIMITE_G_LIQUIDO}g)",
        annotation_position="top right", annotation_font_color="firebrick"
    )

    # Linha Limite Sólido (Cinza)
    fig_slosh.add_hline(
        y=LIMITE_G_SOLIDO,
        line_dash="dot", line_color="gray",
        annotation_text=f"Limite Carga SECA ({LIMITE_G_SOLIDO}g)",
        annotation_position="top right"
    )

    fig_slosh.update_layout(
        title=f"<b>Risco de Tombamento (Slosh): Km {KM_INICIO} ao {KM_FIM}</b><br><sup>Velocidade Constante: {VELOCIDADE_CURVA_KMH} km/h</sup>",
        yaxis_title="Força G Lateral",
        xaxis_title="Distância (km)",
        plot_bgcolor='white', height=500, hovermode="x unified", showlegend=False
    )

    fig_slosh.update_yaxes(showgrid=True, gridcolor='#eee', range=[0, max(0.65, max_g * 1.1)])
    fig_slosh.update_xaxes(showgrid=True, gridcolor='#eee')

    fig_slosh.show()

7️⃣ Calculando Risco de Tombamento (Slosh)...

🌊 ANÁLISE DE SLOSH (40 km/h):
   🌀 Força G Máxima: 1.50g
   ⚠️ PERIGO: 1.50g (Limite Líquido: 0.28g)


## KML

In [ ]:
# ==============================================================================
# 10. GERAÇÃO DE KML COMPLETO (Sinuosidade Original + Safety II + Correção)
# ==============================================================================
print("8️⃣ Gerando KML Master (Fusion: Original + Novo)...")

# --- CONFIGURAÇÕES ---
NOME_KML_SAIDA = "Laudo_Master_SafetyII.kml"
MASSA_REF = 45 # Toneladas

# --- 1. RESGATE DO CÁLCULO ORIGINAL DE SINUOSIDADE ---
# (Essa lógica estava no seu primeiro script)
def haversine_calc(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2) * np.sin(dlambda/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

sinuosidade = []
window_sin = 10
for i in range(len(dist_trecho)):
    s = max(0, i - window_sin)
    e = min(len(dist_trecho) - 1, i + window_sin)

    dist_real = (dist_trecho[e] - dist_trecho[s]) * 1000 # Convertendo km p/ m
    dist_reta = haversine_calc(lats_trecho[s], lons_trecho[s], lats_trecho[e], lons_trecho[e])

    if dist_reta > 0:
        idx = dist_real / dist_reta
    else:
        idx = 1.0
    sinuosidade.append(idx)

# Suaviza a sinuosidade (igual ao código original)
sin_smooth = savgol_filter(sinuosidade, 21, 2)

# --- 2. CORREÇÃO DE INCLINAÇÃO (REALISMO) ---
# Limita a inclinação para evitar leitura de "precipício" do satélite
# Mantém entre -18% e +18% (Padrão rodoviário extremo)
grade_realista = np.clip(grade_smooth, -18.0, 18.0)

# --- 3. GERAÇÃO DO KML ---
kml_header = """<?xml version="1.0" encoding="UTF-8"?>
<kml xmlns="http://www.opengis.net/kml/2.2">
<Document>
    <name>Laudo Safety II - Completo</name>

    <Style id="style_perigo">
        <IconStyle>
            <scale>1.3</scale>
            <Icon><href>http://maps.google.com/mapfiles/kml/shapes/caution.png</href></Icon>
            <color>ff0000ff</color>
        </IconStyle>
        <LabelStyle><scale>0.8</scale></LabelStyle>
        <BalloonStyle>
            <bgColor>ffffffff</bgColor>
            <text>$[description]</text>
        </BalloonStyle>
    </Style>

    <Style id="style_normal">
        <IconStyle>
            <scale>0.6</scale>
            <Icon><href>http://maps.google.com/mapfiles/kml/paddle/blu-circle.png</href></Icon>
        </IconStyle>
        <LabelStyle><scale>0</scale></LabelStyle>
        <BalloonStyle>
            <bgColor>ffffffff</bgColor>
            <text>$[description]</text>
        </BalloonStyle>
    </Style>

    <Style id="linha_rota">
        <LineStyle>
            <color>ff00aaff</color>
            <width>5</width>
        </LineStyle>
    </Style>
"""

kml_body = ""

# Traçado
coords_str = " ".join([f"{lon},{lat},0" for lat, lon in zip(lats_trecho, lons_trecho)])
kml_body += f"""<Placemark><name>Traçado</name><styleUrl>#linha_rota</styleUrl><LineString><coordinates>{coords_str}</coordinates></LineString></Placemark>"""

# Verificações
if 'demanda_freio' not in globals(): demanda_freio = np.zeros(len(dist_trecho))
if 'g_force' not in globals(): g_force = np.zeros(len(dist_trecho))

ultimo_km_plotado = -100

# Loop de Pontos
for i in range(0, len(dist_trecho), 2):

    km_atual = dist_trecho[i]
    alt_atual = alts_plot[i]
    grade_viz = grade_realista[i] # Dado Corrigido
    sin_viz = sin_smooth[i]       # Dado Resgatado
    kw_viz = demanda_freio[i]
    g_viz = g_force[i]

    # Lógica de Risco
    eh_critico_freio = kw_viz > 400
    eh_critico_slosh = g_viz > 0.28
    eh_critico_sinuosidade = sin_viz > 2.5 # Se for muito tortuosa

    eh_critico = eh_critico_freio or eh_critico_slosh or eh_critico_sinuosidade

    distancia_do_ultimo = km_atual - ultimo_km_plotado

    deve_plotar = False
    estilo = "#style_normal"
    nome_ponto = f"Km {km_atual:.1f}"

    if eh_critico:
        if distancia_do_ultimo > 0.1: # 100m
            deve_plotar = True
            estilo = "#style_perigo"
            nome_ponto = f"🚨 Km {km_atual:.1f}"
    else:
        if distancia_do_ultimo > 1.0: # 1km
            deve_plotar = True
            estilo = "#style_normal"

    if deve_plotar:
        ultimo_km_plotado = km_atual

        # --- PREPARAÇÃO DO CARD (IGUAL FOTO ORIGINAL) ---
        if eh_critico_slosh:
            box_bg, box_border = "#ffebee", "#d32f2f"
            box_title = "🔴 CRÍTICO (CURVA/SLOSH)"
            box_desc = f"Risco de Tombamento.<br>Força G: <b>{g_viz:.2f}g</b>"
        elif eh_critico_freio:
            box_bg, box_border = "#ffebee", "#d32f2f"
            box_title = "🔴 CRÍTICO (FREIO)"
            box_desc = f"Superaquecimento.<br>Demanda: <b>{kw_viz:.0f}kW</b>"
        elif eh_critico_sinuosidade:
            box_bg, box_border = "#fff3e0", "#f57c00" # Laranja
            box_title = "🟠 ATENÇÃO (TRAÇADO)"
            box_desc = f"Alta Sinuosidade ({sin_viz:.2f}). Reduza a velocidade."
        else:
            box_bg, box_border = "#f5f5f5", "#9e9e9e"
            box_title = "🔵 MONITORAMENTO"
            box_desc = "Parâmetros normais."

        # Cores numéricas
        c_g = "red" if g_viz > 0.28 else "black"
        c_kw = "red" if kw_viz > 400 else "black"
        c_sin = "orange" if sin_viz > 2.5 else "black"

        html_content = f"""<![CDATA[
            <div style='font-family:Arial, sans-serif; min-width:300px; color:#333;'>
                <div style='border-bottom:2px solid #333; margin-bottom:10px;'>
                    <h2 style='margin:0; font-size:16px;'>Raio-X Km {km_atual:.1f}</h2>
                </div>

                <table style='width:100%; border-collapse:collapse; font-size:12px;'>
                    <tr style='border-bottom:1px solid #ddd;'><td style='color:#666;'>Altitude:</td><td style='font-weight:bold; text-align:right;'>{alt_atual:.0f} m</td></tr>
                    <tr style='border-bottom:1px solid #ddd;'><td style='color:#666;'>Sinuosidade:</td><td style='font-weight:bold; text-align:right; color:{c_sin};'>{sin_viz:.2f}</td></tr>
                    <tr style='border-bottom:1px solid #ddd;'><td style='color:#666;'>Inclinação:</td><td style='font-weight:bold; text-align:right;'>{grade_viz:.1f}%</td></tr>
                    <tr style='border-bottom:1px solid #ddd;'><td style='color:#666;'>Força G:</td><td style='font-weight:bold; text-align:right; color:{c_g};'>{g_viz:.2f} g</td></tr>
                    <tr><td style='color:#666;'>Energia Freio:</td><td style='font-weight:bold; text-align:right; color:{c_kw};'>{kw_viz:.0f} kW</td></tr>
                </table>

                <div style='margin-top:10px; background-color:{box_bg}; border:1px solid {box_border}; padding:8px; border-radius:4px;'>
                    <div style='font-weight:bold; color:{box_border}; font-size:12px;'>{box_title}</div>
                    <div style='font-size:11px; margin-top:4px;'>{box_desc}</div>
                </div>
            </div>
        ]]>"""

        kml_body += f"""
        <Placemark>
            <name>{nome_ponto}</name>
            <styleUrl>{estilo}</styleUrl>
            <description>{html_content}</description>
            <Point><coordinates>{lons_trecho[i]},{lats_trecho[i]},0</coordinates></Point>
        </Placemark>
        """

kml_final = kml_header + kml_body + "</Document></kml>"

path_saida = f"/content/{NOME_KML_SAIDA}"
with open(path_saida, "w", encoding='utf-8') as f:
    f.write(kml_final)

print(f"✅ KML Master Gerado: {path_saida}")
from google.colab import files
files.download(path_saida)

8️⃣ Gerando KML Master (Fusion: Original + Novo)...
✅ KML Master Gerado: /content/Laudo_Master_SafetyII.kml


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==============================================================================
# 9. ANÁLISE DE ESTABILIDADE: CARGA LÍQUIDA (SLOSHING EFFECT)
# ==============================================================================
print("🧪 Analisando Estabilidade de Carga Líquida...")

import numpy as np
import plotly.graph_objects as go

if 'raio_curva' not in globals():
    print("❌ ERRO: Cálculo de RAIOS de curva não encontrado.")
else:
    # --- 1. PARÂMETROS DA CARGA (EDITE AQUI) ---
    VELOCIDADE_KMH = 40          # Velocidade que queremos testar
    NIVEL_ENCHIMENTO = 0.7       # 70% cheio (O mais perigoso pelo balanço)|
    LARGURA_EIXO_M = 2.5         # Distância entre pneus
    ALTURA_CG_M = 2.2            # Altura do Centro de Gravidade do tanque

    # Fator de Redução por Balanço (Slosh Factor)
    # Se estiver 100% cheio ou vazio, o fator é 1.0. Se estiver meio cheio, cai para 0.75.
    fator_estabilidade = 1.0 - (0.25 * (1 - abs(NIVEL_ENCHIMENTO - 0.5) * 2))

    # --- 2. CÁLCULOS ---
    v_ms = VELOCIDADE_KMH / 3.6

    # Limite de Tombamento Teórico (Estático)
    # Segue a regra: Lado / (2 * Altura)
    limite_estatico = (LARGURA_EIXO_M / (2 * ALTURA_CG_M)) * fator_estabilidade

    # Aceleração Lateral em cada ponto da via (g)
    # raio_curva vem da sua análise anterior
    aceleracao_lat_g = (v_ms**2) / (raio_curva * 9.81)

    # Índice de Risco (0 a 1) - 1.0 é o tombo iminente
    risco_tombamento = aceleracao_lat_g / limite_estatico

    # --- 3. TOOLTIP PERSONALIZADO ---
    texto_risco = []
    for r in risco_tombamento:
        if r < 0.5: status = "SEGURO"
        elif r < 0.8: status = "ATENÇÃO: Carga Oscilando"
        else: status = "PERIGO CRÍTICO: Risco de Tombo"
        texto_risco.append(f"Status: {status}<br>Risco: {r*100:.1f}% da capacidade")

    # --- 4. GRÁFICO DE ESTABILIDADE ---
    fig_liq = go.Figure()

    fig_liq.add_trace(go.Scatter(
        x=dist_trecho, y=risco_tombamento * 100,
        mode='lines', name='Risco de Tombamento (%)',
        line=dict(color='purple', width=2),
        fill='tozeroy',
        fillcolor='rgba(128, 0, 128, 0.2)',
        customdata=texto_risco,
        hovertemplate='<b>Km %{x:.1f}</b><br>%{customdata}<extra></extra>'
    ))

    # Linhas de Alerta
    fig_liq.add_hline(y=80, line_dash="dash", line_color="orange", annotation_text="Limite de Segurança")
    fig_liq.add_hline(y=100, line_dash="solid", line_color="red", annotation_text="PONTO DE TOMBAMENTO")

    fig_liq.update_layout(
        title=f"<b>Estabilidade de Carga Líquida ({NIVEL_ENCHIMENTO*100:.0f}% cheio) @ {VELOCIDADE_KMH}km/h</b>",
        yaxis_title="Risco de Tombamento (%)",
        xaxis_title="Distância (km)",
        plot_bgcolor='white', height=500
    )
    fig_liq.update_yaxes(range=[0, 150]) # Escala até 150% para ver o perigo passar do limite
    fig_liq.show()